# Eco-Tracker Data EDA & Verification
이 노트북은 전처리된 YOLO 라벨 데이터가 원본 이미지의 객체 위치와 정확히 일치하는지 시각화하여 검증하기 위해 사용됩니다.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os
import numpy as np
from pathlib import Path
import yaml

In [ ]:
# 클래스 매핑 정보 불러오기
with open('../data/dataset.yaml', 'r', encoding='utf-8') as f:
    dataset_info = yaml.safe_load(f)
class_names = dataset_info['names']
print("Classes:", class_names)

In [ ]:
# 경로 설정 (샘플 데이터 기준)
label_dir = Path('../data/sample_converted/labels')
image_dir = Path('../data/sample/Sample/01.원천데이터')

label_files = list(label_dir.glob('*.txt'))
print(f"Found {len(label_files)} label files.")

In [ ]:
# 시각화 함수
def visualize_bbox(label_file_path, img_dir, classes):
    img_name = label_file_path.stem + '.jpg'
    # 원천데이터 하위 폴더에서 해당 이름의 이미지 찾기
    img_path = next(img_dir.rglob(img_name), None)
    
    if not img_path:
        print(f"Image not found for: {img_name}")
        return
        
    # 한글 경로 지원을 위해 numpy로 파일을 읽어와서 디코딩
    img_array = np.fromfile(str(img_path), np.uint8)
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    
    if img is None:
        print(f"Failed to load image: {img_path}")
        return
        
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    with open(label_file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5: continue
        
        class_id = int(parts[0])
        cx, cy, bw, bh = map(float, parts[1:])
        
        # YOLO (정규화된 중심좌표, 너비/높이) -> 원본 좌표 (좌상단, 우하단)
        x1 = int((cx - bw / 2) * w)
        y1 = int((cy - bh / 2) * h)
        x2 = int((cx + bw / 2) * w)
        y2 = int((cy + bh / 2) * h)
        
        class_name = classes.get(class_id, str(class_id))
        
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 5)
        cv2.putText(img, class_name, (x1, y1 - 15), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 5)
        
    plt.figure(figsize=(12, 12))
    plt.imshow(img)
    plt.title(img_name)
    plt.axis('off')
    plt.show()

# 변환된 라벨 중 3개 샘플 시각화
for label_file in label_files[:3]:
    visualize_bbox(label_file, image_dir, class_names)